# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

# %matplotlib widget

In [ ]:
from rich import print
from tpvalidator.workspace import TriggerActivityWorkspace

import mplhep as hep
import matplotlib.pyplot as plt
import matplotlib as mpl
import mplhep as hep

# Data

In [ ]:
# taws = TriggerActivityWorkspace('../../../radbkg_5000.root', 'taFinder')
# taws = TriggerActivityWorkspace('../../../data/vd/1x8x14/ta_finder/ta-finder_vd-1x8x14-radbkg.root', 'taFinder')
# taws = TriggerActivityWorkspace('../../../data/vd/1x8x14/ta_finder/ta-finder_vd-1x8x14-eminus.root', 'taFinder')
# taws = TriggerActivityWorkspace('../../../data/vd/1x8x14/ta_finder/ta-finder_vd-1x8x14-eminus_nobkg.root', 'taFinder')
# taws = TriggerActivityWorkspace('../../../data/vd/1x8x14/ta_finder/ta-finder_vd-1x8x14-eminus_3tps.root', 'taFinder')
taws = TriggerActivityWorkspace('../../../data/vd/1x8x14/ta_finder/eminus-100k_0000.root', 'taFinder')

print(taws.tree_names)

# Analysis

In [ ]:

import colorsys


# ── Colormap factory ──────────────────────────────────────────────────────────
 
def make_qualitative_ncolors(
    n: int = 16,
    hue_start: float = 0.02,
    sat_dark: float = 0.58,
    sat_light: float = 0.48,
    light_dark: float = 0.44,
    light_light: float = 0.70,
) -> mpl.colors.ListedColormap:
    """
    Build a cyclic qualitative ListedColormap with *n* entries (default 16),
    tuned to tab20-style soft, plot-friendly tones.
 
    Parameters
    ----------
    n           : Number of colours (and the period of the map).
    hue_start   : Starting hue in [0, 1) — 0.02 ≈ 7° offset from pure red.
    sat_dark    : Saturation for even (darker) indices.
    sat_light   : Saturation for odd  (lighter) indices.
    light_dark  : Lightness for even (darker) indices.
    light_light : Lightness for odd  (lighter) indices.
 
    Returns
    -------
    mpl.colors.ListedColormap
    """
    colors = []
    for i in range(n):
        h = (hue_start + i / n) % 1.0
        if i % 2 == 0:
            rgb = colorsys.hls_to_rgb(h, light_dark,  sat_dark)
        else:
            rgb = colorsys.hls_to_rgb(h, light_light, sat_light)
        colors.append(rgb)
    return mpl.colors.ListedColormap(colors, name="qualitative16")
 
 
def _to_hex(rgb_triple) -> str:
    r, g, b = rgb_triple
    return "#{:02x}{:02x}{:02x}".format(int(r * 255), int(g * 255), int(b * 255))


cmap_q16 = make_qualitative_ncolors(10)

display(cmap_q16)


In [ ]:
from typing import Literal
import numpy as np
import hist
from scipy.stats import binom


def histogram_ratio(
    h_num: hist.Hist,
    h_den: hist.Hist,
    mode: Literal["independent", "binomial", "clopper-pearson"] = "independent",
    cl: float = 0.68,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute the ratio of two histograms with appropriate uncertainty.

    Parameters
    ----------
    h_num : hist.Hist
        Numerator histogram (Weight() storage).
    h_den : hist.Hist
        Denominator histogram (Weight() storage).
    mode : str
        - "independent"      : standard error propagation (two uncorrelated samples)
        - "binomial"         : symmetric binomial errors (numerator is subset of denominator)
        - "clopper-pearson"  : exact binomial interval, asymmetric, always within [0, 1]
    cl : float
        Confidence level for clopper-pearson intervals (default 0.68 ≈ 1σ).

    Returns
    -------
    ratio    : np.ndarray  (nan where undefined)
    err_lo   : np.ndarray  lower uncertainty
    err_hi   : np.ndarray  upper uncertainty (== err_lo except for clopper-pearson)
    """
    a, b   = h_num.values(), h_den.values()
    va, vb = h_num.variances(), h_den.variances()

    if va is None or vb is None:
        raise ValueError("Both histograms must use Weight() storage.")

    mask = (a >= 0) & (b > 0)

    with np.errstate(invalid="ignore", divide="ignore"):

        ratio = np.where(mask, a / b, np.nan)

        if mode == "independent":
            err = np.where(mask, ratio * np.sqrt(va / a**2 + vb / b**2), np.nan)
            # guard against a == 0 (ratio is 0, err reduces to sqrt(vb)/b)
            zero_num = mask & (a == 0)
            err = np.where(zero_num, np.sqrt(vb) / b, err)
            err_lo = err_hi = err

        elif mode == "binomial":
            err = np.where(mask, np.sqrt(ratio * (1 - ratio) / b), np.nan)
            err_lo = err_hi = err

        elif mode == "clopper-pearson":
            alpha = 1 - cl
            # use raw counts from values() as k and n
            k, n = a, b
            lo = np.where(mask, binom.ppf(    alpha / 2, n, ratio) / n, np.nan)
            hi = np.where(mask, binom.ppf(1 - alpha / 2, n, ratio) / n, np.nan)
            err_lo = np.where(mask, ratio - lo, np.nan)
            err_hi = np.where(mask, hi - ratio, np.nan)

        else:
            raise ValueError(f"Unknown mode '{mode}'. Choose from: independent, binomial, clopper-pearson.")

    return ratio, err_lo, err_hi



from tpvalidator.analysis.base import TrgWorkspaceAnalyzer
class SwiftTASignalAnalyzer(TrgWorkspaceAnalyzer):
    ...



In [ ]:
taws.info

In [ ]:
swsa = SwiftTASignalAnalyzer(taws)



In [ ]:
swsa.simulated_readout_time()

In [ ]:
taws.ta_event_selection

In [ ]:
taws.mctruths
# [['event','run','subrun', 'kinetic_energy']]

In [ ]:
x = taws.ta_event_selection.merge(taws.mctruths[['event','run','subrun','event_uid','kinetic_energy']], on='event_uid')



In [ ]:
(x[x.accepted == True].kinetic_energy * 1000).hist(bins=150)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1,1, figsize=(10,10))

taws.ta_event_selection.hist(ax=ax, bins=20)


fig.tight_layout()

In [ ]:
taws.event_summary[['event_uid', 'event', 'run', 'subrun', 'tot_visible_energy_rop2']]
# ke_df = aws.mctruths[['event_uid', 'event', 'run', 'subrun', 'kinetic_energy','x', 'y', 'z']]

ev_sum = taws.event_summary[['event_uid', 'event', 'run', 'subrun', 'tot_visible_energy_rop2']].set_index('event_uid')
ev_sum = ev_sum.join(taws.mctruths[['event_uid', 'kinetic_energy','x', 'y', 'z']].set_index('event_uid'))
ev_sum = ev_sum.join(taws.ta_event_selection.drop(columns=['event', 'run', 'subrun']).set_index('event_uid'))
ev_sum['kinetic_energy'] *= 1000

In [ ]:
taws.info

In [ ]:
energy_var = 'kinetic_energy'
label='$E_{kin}$'

energy_var = 'tot_visible_energy_rop2'
label='$E_{vis}$'

from tpvalidator.analysis.histograms import make_regaxis
ke_ax = make_regaxis(ev_sum, energy_var, 1, label=label)


# ke_ax = hist.axis.Regular(99, 1, 100, name=label, flow=False)

h_ke = hist.Hist(ke_ax, storage=hist.storage.Weight())
h_ke.fill(ev_sum[energy_var])

h_ke_acc = hist.Hist(ke_ax, storage=hist.storage.Weight())
h_ke_acc.fill(ev_sum.query('accepted == True')[energy_var])

h_ke_rej = hist.Hist(ke_ax, storage=hist.storage.Weight())
h_ke_rej.fill(ev_sum.query('accepted == False')[energy_var])

h_ke_dir_acc = hist.Hist(ke_ax, storage=hist.storage.Weight())
h_ke_dir_acc.fill(ev_sum.query('num_accept_win > 0')[energy_var])

h_ke_insp_acc = hist.Hist(ke_ax, storage=hist.storage.Weight())
h_ke_insp_acc.fill(ev_sum.query('num_accept_win == 0 & num_inspect_accept_win > 0')[energy_var])

h_ke_insp_rej = hist.Hist(ke_ax, storage=hist.storage.Weight())
# h_ke_insp_rej.fill(ev_sum.query('num_accept_win == 0 & num_inspect_win > 0 & max_win_cluster_sadc < 7500')[energy_var])
h_ke_insp_rej.fill(ev_sum.query('num_accept_win == 0 & num_inspect_win > 0 & num_inspect_accept_win == 0')[energy_var])

# r = (h_ke_acc/h_ke)

# print(histogram_ratio(h_ke_acc, h_ke, 'clopper-pearson'))


# xmin, xmax = 0, 10
xmin, xmax = 0, 50
# xmin, xmax = None, None
cmap = mpl.colormaps['tab10']

fig,axes = plt.subplots(3,3, figsize=(12,12))

# All events
ax=axes[0][0]
hep.histplot(h_ke, ax=ax, color='k')
ax.grid()
ax.set_title('all events')
ax.set_xlim(xmin, xmax)

ax=axes[0][1]
hep.histplot(h_ke_acc, color=cmap(2), ax=ax)
ax.grid()
ax.set_title('accepted - final')
ax.set_xlim(xmin, xmax)

ax=axes[0][2]
# hep.histplot(r, ax=ax)
hep.comp.comparison(h_ke_acc, h_ke, comparison='efficiency', ax=ax, color=cmap(0), linestyle="-")
ax.grid()
ax.set_ylim(0,1.1)
ax.set_title('accepted - efficiency')
ax.set_xlim(xmin, xmax)


ax=axes[1,0]
hep.histplot(h_ke_rej, color=cmap(3), ax=ax)
ax.grid()
ax.set_title('rejected - final')
ax.set_xlim(xmin, xmax)

ax=axes[1,1]
hep.histplot(h_ke_dir_acc, color=cmap(2), ax=ax)
ax.grid()
ax.set_title('direct accept')
ax.set_xlim(xmin, xmax)

ax=axes[1,2]
hep.comp.comparison(h_ke_dir_acc, h_ke, comparison='efficiency', ax=ax, color=cmap(0), linestyle="-")
ax.grid()
ax.set_ylim(0,1.1)
ax.set_title('direct accept - efficiency')
ax.set_xlim(xmin, xmax)

ax=axes[2,0]
hep.histplot(h_ke_insp_rej, color=cmap(3), ax=ax)
ax.grid()
ax.set_title('inspect reject')
ax.set_xlim(xmin, xmax)

ax=axes[2,1]
hep.histplot(h_ke_insp_acc,color=cmap(2),  ax=ax)
ax.grid()
ax.set_title('inspect accept')
ax.set_xlim(xmin, xmax)


ax=axes[2,2]
hep.comp.comparison(h_ke_insp_acc, h_ke, comparison='efficiency', ax=ax, color=cmap(0), linestyle="-")
# hep.histplot(h_ke_insp_acc/h_ke, ax=ax)
ax.grid()
ax.set_title('inspect accept - efficiency')
ax.set_ylim(0,1.1)
ax.set_xlim(xmin, xmax)
ax.set_xlabel(label)

fig.tight_layout()



In [ ]:
from mplhep.comp import get_efficiency, comparison

eff, err = get_efficiency(h_ke_acc, h_ke)


fig, ax = plt.subplots()
comparison(h_ke_acc, h_ke, comparison='efficiency', ax=ax, color=cmap(0), comparison_ylim=(0, 1.1))
ax.grid()



In [ ]:
ev_sum.query('num_accept_win == 0 & num_inspect_win > 0 & num_inspect_accept_win > 0').query('tot_visible_energy_rop2 > 25')

In [ ]:
ev_sum.query('num_accept_win == 0 & num_inspect_win > 0 & num_inspect_accept_win == 0').query('tot_visible_energy_rop2 > 25')

In [ ]:
taws.tree_names

In [ ]:
from tpvalidator.viz.display import TriggerPrimitivesEventViewer

In [ ]:
import tpvalidator.datacatalogue as dctl

em_ws =dctl.load('data/vd/1x8x14/tp_filtered', ['eminus-100k'])['eminus-100k']

In [ ]:
tpv = TriggerPrimitivesEventViewer(em_ws)



In [ ]:
ev_uid = 21000008394
# ev_uid = 21000000007
ev_uid = 21000004357
ev_uid = 22000000013


# tpv.draw_tps_point_of_origin(ev_uid, figsize=(10,10))
tpv.draw_tps_point_of_origin(ev_uid)


# tpv.plot_yzt_event_view(ev_uid)


In [ ]:
ev_sum.query('num_accept_win == 0 & num_inspect_accept_win == 0')

In [ ]:
from tpvalidator.detgeometry import get_by_geocfg_id
import pandas as pd

geo = get_by_geocfg_id(em_ws.info['geo']['detector'])

def decorate_tpc_coords(tps, detgeo):

    tps['tpc_view_channel'] = tps.channel.apply(lambda c: detgeo.tpc_view_channel(c)[1]).astype('int16')
    tps[['tpc_i', 'tpc_k']] = pd.DataFrame(tps.TPCSetID.apply(detgeo.tpc_id_to_grid).tolist(), index=tps.index).rename({0:'tpc_i', 1:'tpc_k'}, axis=1)
    tps['tpc_z_channel'] = (tps['tpc_view_channel']+detgeo.tpc_view_2_num_chans_sim*tps['tpc_k']).where(tps['readout_view'] == 2, -1)


def draw_tawin_boundaries(ax):

    ymin, ymax = ax.get_ylim()
    for k in range(0, geo.tpc_geo[2]*geo.tpc_view_2_num_chans_sim, geo.tpc_view_2_num_chans_sim):
        if k < ymin or k > ymax:
            continue
        ax.axhline(k, c='darkgrey', lw=0.5)

    xmin, xmax = ax.get_xlim()
    for t in range(0, 8500, 1000):
        # print(t)
        if t < xmin or t > xmax:
            continue
        ax.axvline(t, c='darkgrey', lw=0.5)


decorate_tpc_coords(taws.tps_with_cluster_flags, geo)
tps_coll_ev = taws.tps_with_cluster_flags.query(f"event_uid=={ev_uid}")


fig, axes = plt.subplots(2,2, figsize=(12, 10))


ax = axes[0][0]

tps_coll_ev.plot.scatter(x='sample_start', y='tpc_z_channel', s=20, marker='x', c='k', ax=ax)
tps_coll_ev.plot.scatter(x='sample_peak', y='tpc_z_channel', s=tps_coll_ev['adc_integral']/200, c='dbscan_label', cmap='rainbow', ax=ax)

ax.hlines(y=tps_coll_ev.tpc_z_channel.values, xmin=tps_coll_ev.sample_start, xmax=(tps_coll_ev.sample_start+tps_coll_ev.samples_over_threshold))

draw_tawin_boundaries(ax)
ax.grid()

ax = axes[0][1]

tps_coll_ev.plot.scatter(x='sample_peak', y='tpc_z_channel', s=tps_coll_ev['adc_integral']/200, c='TPCSetID', cmap='tab20', ax=ax)

draw_tawin_boundaries(ax)
ax.grid()


ax = axes[1][0]

tps_coll_ev.plot.scatter(x='sample_peak', y='tpc_z_channel', s=tps_coll_ev['adc_integral']/200, c='ta_win_id', cmap='Paired', ax=ax)

draw_tawin_boundaries(ax)
ax.grid()

ax = axes[1][1]

tps_coll_ev.plot.scatter(x='sample_peak', y='tpc_z_channel', s=tps_coll_ev['adc_integral']/200, c='tpc_i', cmap='Set1', ax=ax)

draw_tawin_boundaries(ax)
ax.grid()

fig.tight_layout()


In [ ]:
from tpvalidator.detgeometry import get_by_geocfg_id
detgeo = get_by_geocfg_id(taws.info['geo']['detector'])


from tpvalidator.viz.geo_dg import plot_geometry_2d, draw_tpc_outlines


plot_geometry_2d(detgeo)

tps_coll_ev.plot.scatter(x='bt_primary_y', y='bt_primary_z', ax=plt.gca())

plt.gca().set_xlim(tps_coll_ev.bt_primary_y.min(), tps_coll_ev.bt_primary_y.max())
plt.gca().set_ylim(tps_coll_ev.bt_primary_z.min(), tps_coll_ev.bt_primary_z.max())

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(10, 10))
tpc_ids = tps_coll_ev.TPCSetID.unique()

sm = draw_tpc_outlines(ax, detgeo, plane='yz', tpc_ids=tpc_ids, c=tpc_ids, colormap='tab20', alpha=0.9, linewidth=2.6)
ax.set_aspect('equal')
ax.autoscale()

tps_coll_ev.plot.scatter(x='bt_primary_y', y='bt_primary_z', c='TPCSetID', colormap='tab20', ax=ax)

In [ ]:
tps_coll_ev

In [ ]:
em_ws.mctruths.query('kinetic_energy < 0.05')


In [ ]:
em_ev = TriggerPrimitivesEventViewer(em_ws)

In [ ]:
datasets=dctl.load('data/vd/1x8x14/preprod', 'eminus-100k')

In [ ]:
x = TriggerPrimitivesEventViewer(datasets['eminus-100k'])

In [ ]:
l = datasets['eminus-100k'].mctruths.query('kinetic_energy*1000 > 5 & kinetic_energy*1000 < 15').event_uid.unique()

In [ ]:
from tpvalidator.utils import subplots_autogrid

n_events = 12

fig, axes = subplots_autogrid(n_events, figsize=(16, 10))

for i,e in enumerate(l[:n_events]):
    x.plot_yzt_event_view(e, ax=axes[i])
    ke = (x.mctruths.query(f'event_uid=={e}').kinetic_energy*1000).iloc[0]
    axes[i].set_title(f"{ke:.1f} MeV")

    fig.tight_layout()

In [ ]:
sel_tps = em_ws.tps.query('bt_is_signal==True & readout_plane_id == 2')

sel_tps = sel_tps[sel_tps.event_uid.isin(em_ws.mctruths[em_ws.mctruths.kinetic_energy < 0.10].event_uid)]

fig, axes = plt.subplots(2,2, figsize=(12, 12))


ax = axes[0][0]
sel_tps.plot.scatter(x='bt_primary_y', y='bt_primary_z', s=0.01, alpha=0.1, ax=ax)
ax.grid()
ax.autoscale()
# ax.set_ylim(140, 165)
# ax.set_xlim(440, 460)

# tps_pp = x.decorate_tpc_coords()

draw_tpc_outlines(ax, detgeo, plane='yz')
ax.set_aspect('equal')
ax.autoscale()

ax = axes[0][1]
sel_tps.plot.scatter(x='bt_primary_y', y='bt_primary_z', s=0.01, alpha=1, ax=ax, c=sel_tps.TPCSetID % 10, cmap='tab10')
ax.grid()
ax.set_xlim(50, 250)
ax.set_ylim(400, 500)

draw_tpc_outlines(ax, detgeo, c='k', plane='yz')
ax.set_aspect('equal')
# ax.autoscale()


ax = axes[1][0]
sel_tps = sel_tps
sel_tps.plot.scatter(x='bt_primary_y', y='bt_primary_z', s=0.01, alpha=1, ax=ax, c=sel_tps.TPCSetID % 10, cmap='tab10')
ax.grid()
# ax.set_ylim(140, 160)
# ax.set_xlim(440, 460)
print(sel_tps.TPCSetID.unique())


tpc_ids = sel_tps.TPCSetID.unique()
draw_tpc_outlines(ax, detgeo, tpc_ids=tpc_ids, c='k', plane='yz')
ax.set_aspect('equal')
ax.autoscale()

n_bins=max(tpc_ids)-min(tpc_ids)+1
sel_tps.TPCSetID.hist(ax=axes[1][1], bins=n_bins)


In [ ]:
tps_not_inspect = em_ws.tps[em_ws.tps.event_uid.isin(ev_sum.query('num_accept_win == 0 & num_inspect_win > 0 & num_inspect_accept_win == 0').index)]

# print(len(em_ws.mctruths.query('kinetic_energy > 0.01 & kinetic_energy < 0.02').event_uid))
# tps_not_inspect = tps_not_inspect[tps_not_inspect.event_uid.isin(em_ws.mctruths.query('kinetic_energy > 0.01').event_uid)]

sel_tps = tps_not_inspect.query('bt_is_signal==True & readout_plane_id == 2')

sel_tps = sel_tps[sel_tps.event_uid.isin(em_ws.mctruths[em_ws.mctruths.kinetic_energy > 0.02].event_uid)]

print('Selection completed')

fig, ax = plt.subplots(1,1, figsize=(8, 8))


# ax = axes[0][0]
# sel_tps.plot.scatter(x='bt_primary_y', y='bt_primary_z', s=0.01, alpha=0.1, ax=ax)
# ax.grid()
# ax.autoscale()
# # ax.set_ylim(140, 165)
# # ax.set_xlim(440, 460)

# # tps_pp = x.decorate_tpc_coords()

# draw_tpc_outlines(ax, detgeo, plane='yz')
# ax.set_aspect('equal')
# ax.autoscale()

# ax = axes[0][1]
# sel_tps.plot.scatter(x='bt_primary_y', y='bt_primary_z', s=0.01, alpha=1, ax=ax, c=sel_tps.TPCSetID % 10, cmap='tab10')
sel_tps.plot.scatter(x='bt_primary_y', y='bt_primary_z', s=0.5, alpha=1, ax=ax, c='bt_primary_x', cmap='turbo')

ax.grid()
ax.set_xlim(-0, 180)
ax.set_ylim(450, 600)

draw_tpc_outlines(ax, detgeo, c='k', plane='yz')
ax.set_aspect('equal')
# ax.autoscale()


# ax = axes[1][0]
# sel_tps = sel_tps
# sel_tps.plot.scatter(x='bt_primary_y', y='bt_primary_z', s=0.01, alpha=1, ax=ax, c=sel_tps.TPCSetID % 10, cmap='tab10')
# ax.grid()
# # ax.set_ylim(140, 160)
# # ax.set_xlim(440, 460)
# print(sel_tps.TPCSetID.unique())


# tpc_ids = sel_tps.TPCSetID.unique()
# draw_tpc_outlines(ax, detgeo, tpc_ids=tpc_ids, c='k', plane='yz')
# ax.set_aspect('equal')
# ax.autoscale()

# n_bins=max(tpc_ids)-min(tpc_ids)+1
# sel_tps.TPCSetID.hist(ax=axes[1][1], bins=n_bins)


In [ ]:
# inspect_evs = taws.ta_event_selection.query("num_inspect_win > 0 & num_accept_win == 0 & num_inspect_accept_win == 0")
inspect_evs = taws.ta_event_selection.query("num_inspect_win > 0 & num_accept_win == 0")

ev_sel_uids = inspect_evs.event_uid 

key_cols = ['event_uid', 'event', 'run', 'subrun']
mc_cols = ['kinetic_energy', 'x']


inspect_evs.max_win_cluster_sadc.hist(bins=100)


df = inspect_evs.merge(taws.mctruths[taws.mctruths.event_uid.isin(ev_sel_uids)][key_cols+mc_cols], on=key_cols)

display(df)

df = df.merge(taws.event_summary[taws.event_summary.event_uid.isin(ev_sel_uids)], on=key_cols)

display(df)

print(taws.event_summary[taws.event_summary.event_uid.isin(inspect_evs.event_uid)].tot_visible_energy_rop2)
print(inspect_evs.max_win_cluster_sadc)

fix, axes = plt.subplots(2, 2, figsize=(12, 10))

# ax.scatter(x=taws.event_summary[taws.event_summary.event_uid.isin(inspect_evs.event_uid)].tot_visible_energy_rop2, y=inspect_evs.max_win_cluster_sadc, s=0.1, c=taws.mctruths[taws.mctruths.event_uid.isin(inspect_evs.event_uid)].x)


df.plot.scatter(x='tot_visible_energy_rop2', y='max_win_cluster_sadc', s=0.1, c='x', ax=axes[0][0])
df.plot.scatter(x='kinetic_energy', y='max_win_cluster_sadc', s=0.1, c='x', ax=axes[0][1])
df.plot.scatter(x='kinetic_energy', y='tot_visible_energy_rop2', s=0.1, c='x', ax=axes[1][0])
# ax.set_ylim(ymax=5000)

fig.tight_layout()

In [ ]:
inspect_evs.query('max_win_cluster_sadc == 0')

In [ ]:
x.plot_yzt_event_view(22000000057)

In [ ]:
def draw_windows_yz(ws, ev_uid, ax):


    ev_wins = ws.ta_win_stats.query(f"event_uid == {ev_uid}")

    # ta_win_ids = ev_wins.ta_win_id.unique()
    # for ta_win_id in sorted(ta_win_ids):
    #     ev_wins.


    draw_tpc_outlines(ax, detgeo, plane='yz', c='k', linewidth='0.1')

    for wid, wins_df in ev_wins.groupby(by='ta_win_id'):
        
        wins_acc = wins_df.query('sadc_window_thres_hi == 1')
        wins_ins = wins_df.query('sadc_window_thres_lo == 1 & sadc_window_thres_hi == 1')

        draw_tpc_outlines(ax, detgeo, plane='yz', tpc_ids=wins_acc.TPCSetID.unique(), c='g', alpha=0.3, fill=True)
        draw_tpc_outlines(ax, detgeo, plane='yz', tpc_ids=wins_ins.TPCSetID.unique(), c='y', alpha=0.3, fill=True)

        ax.autoscale()
        ax.set_aspect('equal')


    ...



ev_uid = 22000001284

fig, ax = plt.subplots(figsize=(8,8))

draw_windows_yz(taws, ev_uid, ax)

em_ws.tps.query(f"event_uid == {ev_uid} & readout_plane_id == 2 & bt_is_signal == True").plot.scatter(x='bt_primary_y', y='bt_primary_z', s=0.5, alpha=1, ax=ax, c='bt_primary_x', cmap='turbo')



In [ ]:
ev_sum.query('num_accept_win == 0 & num_inspect_win > 0 & num_inspect_accept_win == 0 & tot_visible_energy_rop2 > 25')

In [ ]:
ev_uid = 22000097110


fig, ax = plt.subplots(figsize=(8,8))


ev_tps = em_ws.tps.query(f"event_uid == {ev_uid} & readout_plane_id == 2 & bt_is_signal == True")
# print(ev_tps.columns)
# draw_windows_yz(taws, ev_uid,ax=ax)
# draw_tpc_outlines(ax, detgeo, plane='yz', tpc_ids=ev_tps.TPCSetID.unique())

ax.set_aspect('equal')

ev_tps.plot.scatter(x='bt_primary_y', y='bt_primary_z', s=ev_tps.adc_peak / 10, alpha=1, ax=ax, c='bt_primary_x', cmap='turbo')
ax.grid()


# Look into diffusion/electron lifetime

In [ ]:
em_ws.mctruths.query('(pz / p > 0.9) & (kinetic_energy < 0.012) & (kinetic_energy > 0.009) & x > 290')

In [ ]:
print('Close to Cathode')
fig = em_ev.plot_yzt_event_view(22000097083)


print('Close to Anode')
fig = em_ev.plot_yzt_event_view(22000085503)

In [ ]:
dx = 25
for x in np.arange(-325, +325, dx):
    em_ws.mctruths.query(f'(pz / p > 0.9) & (kinetic_energy < 0.012) & (kinetic_energy > 0.009) & x > {x} & x < {x+dx}')


In [ ]:


em_ws.mctruths.query('(pz / p > 0.9) & (kinetic_energy < 0.012) & (kinetic_energy > 0.009) & x > 290')